# Imports

In [ ]:
# Purpose: standard imports and project path setup
import dataclasses
import itertools
import json
import math
import os
import random
import sys
import time
from itertools import product
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from matplotlib import cm
from matplotlib import colors as mcolors
from matplotlib.dates import DateFormatter
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src import features, training
from src import training as training

# Add project root
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path: sys.path.append(ROOT)

from src import features as features
from src import models as models
from src import preprocessing as preprocessing
from src import training as training
from src import visualisation as viz

print("Torch:", torch.__version__)

# Configuration

In [ ]:
# Purpose: define config, seeds, and (optionally) exam periods for stratification
cfg = preprocessing.Config(
    time_granularity="15min",
    H_in=12, H_out=12,
    batch_size=64, lr=1e-3,
    max_epochs=30, patience=8,
    seed=42, use_exog=True,
    gcn_type="cheb", cheb_K=3, nblocks=2, hidden=32, dropout=0.12,
    lambda_lap=1e-3
)
debug = False

# grid search settings
feature_grid_search = False          # takes a long time if True
hyperparam_optimisation = False      # takes a long time if True

features.set_seed(cfg.seed)

# Define exam ranges and holiday dates
exam_ranges = [
    ("2025-01-20","2025-01-31"),
    ("2025-04-07","2025-04-17"),
]
holiday_dates = [
    ("2025-01-01","2025-01-05"),
    ("2025-02-01","2025-02-09"),
    ("2025-04-18","2025-04-21"),
]


In [ ]:
if debug:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # sync CUDA for clearer stacktraces
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("CUDA available:", torch.cuda.is_available(), "device in cfg:", cfg.device)

#### Graph Network

In [ ]:
# Nodes
nodes = ['A','B','C','D','E','G','H','I','J','K','L','M','N','O','P']

# Example coordinates in (x, y). Replace with your true values (meters or canvas units).
pos = {
    'A': (0, 10), 'B': (5, 10),
    'C': (0, 7),  'D': (5, 7),
    'E': (0, 4), 'G': (9, 4),
    'H': (-3, 1), 'I': (0, 1), 'J': (5, 1), 'K': (9, 1), 'L': (12, 1),
    'M': (0, -2), 'N': (5, -2), 'O': (9, -2),
    'P': (5, -5)
}

# Physical undirected links
physical_edges = [
    ('A','B'),        # no sensor
    ('A','C'),
    ('C','E'),
    ('B','D'),
    ('D','J'),
    ('C','D'),        # no sensor
    ('D','G'),        # no sensor
    ('G','K'),
    ('K','O'),
    ('H','I'),
    ('I','J'),
    ('J','K'),
    ('K','L'),
    ('E','I'),
    ('J','N'),        # no sensor
    ('N','P'),
    ('I','M'),
    ('M','N'),        # no sensor
    ('N','O')         # no sensor
]

# Edges without sensors (undirected listing). Both directions will be masked.
no_sensor_undir = {('A','B'), ('M','N'), ('N','O'), ('D','G'), ('J','N'), ('C','D')}

# Map each (sensor_id|element_name) to its BASE direction (u->v).
# count_in goes to (u->v); count_out goes to (v->u).
sensor_to_edge: Dict[str, Tuple[str,str]] = {
    "TUD_SBX01|Line 0": ("D", "B"),
    "TUD_SPX18|Line 0": ("A", "C"),
    "TUD_SBX22|Line 0": ("E", "C"),
    "TUD_SBXN6A|Line 2": ("D", "J"),
    "TUD_SBXN6A|Line 0": ("J", "K"),
    "TUD_SPX23|Line 0": ("I", "E"),
    "TUD_SBX25|Line 2": ("I", "H"),
    "TUD_SBX26|Line 0": ("I", "M"),
    "TUD_SPX24|Line 0": ("J", "I"),
    "TUD_SBX07-MULTI|Line 1": ("G", "K"),
    "TUD_SBX07-MULTI|Line 2": ("K", "L"),
    "TUD_SBX07-MULTI|Line 3": ("O", "K"),
    "TUD_SBXN13|Line 0": ("P", "N"),
}

In [ ]:
directed_edges = preprocessing.undir_to_dir(physical_edges)
edge_index = {e:i for i,e in enumerate(directed_edges)}
index_edge = {i:e for e,i in edge_index.items()}

A_line = preprocessing.build_line_graph_adjacency(directed_edges, allow_uturn=False)
# choose ONE normalization policy; we keep row-normalized with self-loops once
A_hat = preprocessing.row_normalize_with_self_loops(A_line)

network_cfg = preprocessing.NetworkConfig(
    nodes=nodes, pos=pos, physical_edges=physical_edges, no_sensor_undir=no_sensor_undir,
    sensor_to_edge=sensor_to_edge, directed_edges=directed_edges,
    edge_index=edge_index, index_edge=index_edge, a_line=A_line, a_hat=A_hat
)

viz.draw_sensor_coverage_network(network_cfg, title="Network: sensor-supervised vs no-sensor")

# Data loading and Preprocessing 

In [ ]:
# Purpose: Build data, exogenous, masks, and gap-aware loaders using src functions

csv_paths = [
    "data/smartcameras_line--2025-01-01T03-01-00--2025-02-01T03-01-00--tud_project--y16mv1.csv",
    "data/smartcameras_line--2025-02-01T03-01-00--2025-03-01T03-01-00--tud_project--bvtmix.csv",
    "data/smartcameras_line--2025-03-01T03-01-00--2025-04-01T03-01-00--tud_project--a93fyx.csv",
    "data/smartcameras_line--2025-04-01T03-01-00--2025-05-01T03-01-00--tud_project--henvbs.csv",
]
nc_paths = [
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202501.nc",
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202502.nc",
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202503.nc",
    "data/davis-TUD-EWI_Roof_Electrical_Engineering_202504.nc",
]

data = preprocessing.build_data_and_loaders(
    csv_paths=csv_paths,
    nc_paths=nc_paths,
    cfg=cfg,
    network_cfg=network_cfg,
    exam_ranges=exam_ranges,
    holiday_ranges=holiday_dates,
)

times      = data["times"]
X, M, T, E = data["X"], data["M"], data["T"], data["E"]
weather_df = data["weather_df"]
exog       = data["exog"]
cats       = data["cats"]
flags      = data["flags"]
masks      = data["masks"]
loaders    = data["loaders"]

print(f"T={T}, E={E}, exog={exog.shape}")
print("weather cols:", None if weather_df is None else list(weather_df.columns))
print("batches:", {k: len(v) for k,v in loaders.items()})
print("lags:", data["lags"])

viz.print_split_day_summary(times, masks,
                            holiday_ranges=holiday_dates,
                            exam_ranges=exam_ranges)


# Feature grid search

In [ ]:
# === Feature selection: full powerset grid (if enabled) OR preset features (if disabled) ===

# ---- 0) Build context_df from columns ----
if 'weather_df' not in globals() or weather_df is None or weather_df.empty:
    print("weather_df is missing or empty; only calendar flags will be considered.")
else:
    print("weather_df columns:", list(weather_df.columns))

context_df = pd.DataFrame(index=pd.DatetimeIndex(times))

wanted_weather = ["temperature", "rain_mm", "rain_mm_h", "wind_mps", "wind_gust_speed"]
present_weather = []
if ('weather_df' in globals()) and (weather_df is not None) and (not weather_df.empty):
    present_weather = [c for c in wanted_weather if c in weather_df.columns]
    if present_weather:
        context_df = pd.concat([context_df,
                                weather_df[present_weather].reindex(context_df.index)],
                               axis=1)

wanted_flags = ["is_weekend","is_exam","is_holiday"]
present_flags = [c for c in wanted_flags if c in flags.columns]
context_df = pd.concat([context_df, flags[present_flags].reindex(context_df.index)], axis=1)

print("Using weather columns:", present_weather)
print("Using calendar flags:", present_flags)

candidate_cols = [c for c in (present_weather + present_flags) if c in context_df.columns]
print(f"Candidate features (N={len(candidate_cols)}): {candidate_cols}")

# ---- 1) Lags & adjacency (always on) + exog builder ----
L_24h = features.steps_for_hours(cfg.time_granularity, 24)
L_7d  = features.steps_for_hours(cfg.time_granularity, 24*7)
A_bin = (network_cfg.a_line > 0).astype(np.float32)

def build_exog_from_subset(selected_cols):
    """Selected extras from context_df; add_exogenous appends lags & TOD/DOW."""
    ctx = context_df[selected_cols] if selected_cols else pd.DataFrame(index=context_df.index)
    exog = features.add_exogenous(
        times=pd.DatetimeIndex(times), E=E, config=cfg,
        weather_df=ctx,              # only the chosen extras (weather+flags by name)
        calendar_flags=None,         # avoid double-adding flags
        X=X, M=M, A_bin=A_bin,
        use_graph_fill=True,
        lags_self=(1,2,3, L_24h, L_7d),
        lags_neigh=(1,2),
        ema_alpha=0.7, fill_weights=(0.7,0.3,0.0),
        Xhat_prev=None
    ).astype("float32")
    return np.nan_to_num(exog, nan=0.0, posinf=0.0, neginf=0.0)

def run_one_combo(selected_cols, budget_epochs=20):
    ex = build_exog_from_subset(selected_cols)
    loaders = training.make_loaders_gapaware(pd.DatetimeIndex(times), X, M, ex, cfg, masks, shuffle_train=True)
    Fin = ex.shape[2] + 1
    model = training.build_gnn(E, Fin, cfg.H_out, network_cfg.a_hat, cfg)
    cfg_try = dataclasses.replace(cfg, max_epochs=budget_epochs)
    # mute per-epoch bars during the grid
    model = training.train_model(model, loaders, cfg_try, epoch_hook=None, A_bin=A_bin,
                                 show_epoch_bar=False, leave_bar=False)
    val = training.evaluate(model, loaders["val"],  cfg_try.device)["MAE"]
    test = training.evaluate(model, loaders["test"], cfg_try.device)["MAE"]
    return float(val), float(test)

# ---- 2) Toggle: run the grid OR pick a preset ----
if feature_grid_search:
    # Full powerset (includes empty set)
    all_combs = [tuple(comb) for r in range(0, len(candidate_cols)+1)
                 for comb in itertools.combinations(candidate_cols, r)]
    print(f"Total settings to evaluate: {len(all_combs)} (2^{len(candidate_cols)})")

    rows = []
    best_val = float("inf")
    best_label = "(none)"
    t0 = time.time()

    for comb in tqdm(all_combs, desc="Feature grid", leave=False):
        label = "(none)" if len(comb)==0 else "+".join(comb)
        mae_val, mae_test = run_one_combo(list(comb), budget_epochs=20)
        rows.append({
            "features_on": label,
            "combo": comb,
            "k_on": len(comb),
            "gnn_val_MAE": mae_val,
            "gnn_test_MAE": mae_test,
        })
        if mae_val < best_val:
            best_val, best_label = mae_val, label
            tqdm.write(f"best so far: {best_label} -> {best_val:.4f}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    res_df = pd.DataFrame(rows).sort_values(["gnn_val_MAE","k_on"]).reset_index(drop=True)
    print(f"Grid done over {len(res_df)} settings in {time.time()-t0:.1f}s")
    display(res_df.head(10))

    best_combo = list(res_df.iloc[0]["combo"])  # list of feature names
    exog_best  = build_exog_from_subset(best_combo)

else:
    # PRESET path: keep downstream variables consistent
    preset_features = ['temperature', 'rain_mm_h', 'wind_gust_speed', 'is_weekend', 'is_exam']  # <- edit to your preferred default
    best_combo = [c for c in preset_features if c in context_df.columns]
    print("Feature grid disabled. Using preset features:", best_combo if best_combo else "(none)")
    exog_best  = build_exog_from_subset(best_combo)

    # Minimal res_df so later cells (that expect res_df) still work
    res_df = pd.DataFrame([{
        "features_on": "(preset)" if best_combo else "(none)",
        "combo": tuple(best_combo),
        "k_on": len(best_combo),
        "gnn_val_MAE": np.nan,
        "gnn_test_MAE": np.nan,
    }])


In [ ]:
# === Single-feature MAE + average-in-combos; Top-5 ===
pd.set_option("display.max_colwidth", None) # don't truncate combo display

def mae_only_feature(df, f):
    if f == "(none)":
        row = df[df["k_on"]==0].head(1)
    else:
        row = df[(df["k_on"]==1) & (df["combo"].apply(lambda c: (len(c)==1 and c[0]==f)))]
    return float(row["gnn_val_MAE"].iloc[0]) if len(row) else float("nan")

def mae_avg_when_included(df, f):
    if f == "(none)":
        row = df[df["k_on"]==0]
        return float(row["gnn_val_MAE"].mean()) if len(row) else float("nan")
    sel = df[df["combo"].apply(lambda c: f in c)]
    return float(sel["gnn_val_MAE"].mean()) if len(sel) else float("nan")

rows = []
rows.append({
    "feature": "(none)",
    "only_val_MAE": mae_only_feature(res_df, "(none)"),
    "avg_val_MAE_when_included": mae_avg_when_included(res_df, "(none)")
})
for f in candidate_cols:
    rows.append({
        "feature": f,
        "only_val_MAE": mae_only_feature(res_df, f),
        "avg_val_MAE_when_included": mae_avg_when_included(res_df, f),
    })
feature_table = pd.DataFrame(rows).sort_values("avg_val_MAE_when_included", ascending=True).reset_index(drop=True)
display(feature_table)

top5 = res_df.sort_values(["gnn_val_MAE","k_on"]).head(5)
display(top5[["features_on","k_on","gnn_val_MAE","gnn_test_MAE"]])

print(f"Evaluated {len(res_df)} combos out of {2**len(candidate_cols)} possible.")


# Grid search for hyperparameters

In [ ]:
# === Hyperparameter grid: resumable + toggleable ===
RESULTS_HP = "artifacts/hp_grid_results.csv"
os.makedirs(os.path.dirname(RESULTS_HP), exist_ok=True)


best_combo = list(res_df.iloc[0]["combo"])
print("Best feature combo:", best_combo if best_combo else "(none)")
exog_best = build_exog_from_subset(best_combo)
A_bin = (network_cfg.a_line > 0).astype(np.float32)

# Search space
hp_space = {
    "time_granularity": ["5min", "15min", "30min"],
    "cheb_K":           [2, 3],
    "hidden":           [32, 48, 64],
    "nblocks":          [2, 3],
    "dropout":          [0.10, 0.20],
    "lr":               [1e-3, 1e-4, 1e-5],
    "weight_decay":     [0.0, 1e-4],
    "patience":         [8, 12],
    "H_in":             [12, 24],
    "H_out":            [12, 24],
}

def run_one_hp(cfg_base, exog, budget_epochs=20):
    loaders = training.make_loaders_gapaware(pd.DatetimeIndex(times), X, M, exog, cfg_base, masks, shuffle_train=True)
    Fin = exog.shape[2] + 1
    model = training.build_gnn(E, Fin, cfg_base.H_out, network_cfg.a_hat, cfg_base)
    # TRAIN: no epoch bar during grid
    cfg_try = dataclasses.replace(cfg_base, max_epochs=budget_epochs)
    model = training.train_model(model, loaders, cfg_try, epoch_hook=None, A_bin=A_bin,
                                 show_epoch_bar=False, leave_bar=False)
    val = training.evaluate(model, loaders["val"], cfg_try.device)["MAE"]
    test = training.evaluate(model, loaders["test"], cfg_try.device)["MAE"]
    return float(val), float(test)

# --- Build combinations from hp_space (assumed defined above) ---
hp_keys = list(hp_space)
hp_vals = [hp_space[k] for k in hp_keys]
combos  = [dict(zip(hp_keys, v)) for v in product(*hp_vals)]
print(f"HP trials: {len(combos)}")

# --- helper: stable key for a combo row ---
def hp_key(d: dict) -> str:
    return json.dumps({k: d[k] for k in sorted(d)}, sort_keys=True)

# --- helper: map aliases (e.g. weight_decay -> wd) and drop unknown fields ---
from dataclasses import fields

_CFG_FIELDS = {f.name for f in fields(cfg)}
_ALIAS = {"weight_decay": "wd"}  # extend if needed
def dataclass_replace_known(c, updates: dict):
    mapped   = {(_ALIAS.get(k, k)): v for k, v in updates.items()}
    filtered = {k: v for k, v in mapped.items() if k in _CFG_FIELDS}
    return dataclasses.replace(c, **filtered)

if hyperparam_optimisation:
    # resume set
    if os.path.exists(RESULTS_HP):
        hp_prev = pd.read_csv(RESULTS_HP)
        tried = set(hp_prev["key"].tolist())
    else:
        tried = set()

    rows = []
    best_val  = float("inf")
    best_test = None
    cfg_best  = None

    pbar = tqdm(combos, desc="HP grid", leave=False)
    for hp in pbar:
        key = hp_key(hp)
        if key in tried:
            continue

        cfg_try = dataclass_replace_known(cfg, hp)
        try:
            val_mae, test_mae = run_one_hp(cfg_try, exog_best, budget_epochs=20)
        except RuntimeError as e:
            print("RuntimeError during HP trial with config:", hp)
            raise e

        rec = {**hp, "key": key, "val_MAE": val_mae, "test_MAE": test_mae}
        rows.append(rec)

        # append immediately (so we can resume after crashes)
        pd.DataFrame([rec]).to_csv(
            RESULTS_HP,
            mode=("a" if os.path.exists(RESULTS_HP) else "w"),
            header=not os.path.exists(RESULTS_HP),
            index=False,
        )

        if val_mae < best_val:
            best_val, best_test, cfg_best = val_mae, test_mae, cfg_try
            pbar.set_postfix(best=f"{best_val:.4f}")

    pbar.close()

    # Final table (load from file, not just memory)
    hp_df = pd.read_csv(RESULTS_HP).sort_values("val_MAE").reset_index(drop=True)
    # Ensure cfg_best defined even if everything was already done before this run
    if cfg_best is None and len(hp_df):
        # reconstruct cfg_best from best row
        best_row = hp_df.iloc[0].to_dict()
        # keep only hp keys
        best_hp = {k: best_row[k] for k in hp_keys if k in best_row}
        cfg_best = dataclass_replace_known(cfg, best_hp)
        best_val  = float(best_row["val_MAE"])
        best_test = float(best_row["test_MAE"])

else:
    hp_df = pd.DataFrame()
    default_hp = {"time_granularity": "15min", "H_in": 12, "H_out": 12, "cheb_K": 2, "dropout": 0.2, "hidden": 64, "lr": 0.00001, "nblocks": 3, "weight_decay": 0.001, "patience": 12}
    print("Hyperparameter optimisation disabled. Using precalculated HPs:", default_hp)
    cfg_best = dataclass_replace_known(cfg, default_hp)
    best_val = best_test = float("nan")

# Summary peek (safe even when skipped)
if len(hp_df):
    display(hp_df.head(10))
print("cfg_best ready. Best val:", best_val, "| test:", best_test)

In [ ]:
# hp_df = pd.DataFrame(rows).sort_values("val_MAE").reset_index(drop=True)
# print(f"HP grid finished in trials={len(hp_df)}")
# display(hp_df.head(10))
# print("Best val MAE:", best_val, " | Corresponding test MAE:", best_test)
# cfg_best

# Training and Prediction

In [ ]:
# === Final comparison: Historical avg vs Linear (with exog_best) vs GNN(best cfg, 2×epochs) ===

# 0) Masks: TRAIN+VAL for final fit (val kept for monitoring/curves)
trainval_mask = masks["train"] | masks["val"]
masks_final = {"train": trainval_mask, "val": masks["val"], "test": masks["test"]}

# 1) Exogenous and cfg from previous cells
#    - exog_best: from the feature-grid winner (lags always on + your chosen extras)
#    - cfg_best:  from the HP grid winner (includes gcn/cheb, K, hidden, etc.)

# 2) Build loaders (gap-aware) with TRAIN+VAL as train
cfg_final = dataclasses.replace(cfg_best, max_epochs=80, H_out=1)
loaders_final = training.make_loaders_gapaware(pd.DatetimeIndex(times), X, M, exog_best, cfg_final, masks_final, shuffle_train=True)

Fin = exog_best.shape[2] + 1
A_bin = (network_cfg.a_line > 0).astype(np.float32)


# Linear regressor with exogenous features
# Uses x[..., 1:] = exog channels averaged over H_in as features.
W_lin, b_lin = training.fit_linear_regressor(loaders_final["train"].dataset)

# Prepare test tensors for metrics (keep your existing Y_fut_test / M_fut_test build)
ds_test = loaders_final["test"].dataset
xb_list, yb_list, mb_list = [], [], []
for i in range(len(ds_test)):
    it = ds_test[i]
    xb_list.append(it["x"].numpy()); yb_list.append(it["y"].numpy()); mb_list.append(it["mask"].numpy())
X_hist_test = np.stack(xb_list, axis=0)
Y_fut_test  = np.stack(yb_list, axis=0)
M_fut_test  = np.stack(mb_list, axis=0)

# Predict horizon-0 and repeat across H_out for metric parity
lin_pred = training.predict_linear_regressor(W_lin, b_lin, loaders_final["test"].dataset)  # [B,1,E]
lin_pred = np.repeat(lin_pred, repeats=Y_fut_test.shape[1], axis=1)  # [B,H_out,E]


# 5) (c) GNN(best) — same extras, best HPs, but 4× the HP-grid budget epochs

pred_hook = training.make_prediction_driven_hook(
    X, M, pd.DatetimeIndex(times), E, cfg_final,
    weather_df=context_df[best_combo] if len(best_combo) else pd.DataFrame(index=pd.DatetimeIndex(times)),
    A_bin=(network_cfg.a_line > 0).astype(np.float32),
    slices=masks_final,
    loaders=loaders_final,
    lags_self=(1,2,3, L_24h, L_7d),
    lags_neigh=(1,2),
    ema_alpha=0.7, fill_weights=(0.5,0.3,0.2),
)

model_best = training.build_gnn(E, Fin, cfg_final.H_out, network_cfg.a_hat, cfg_final)
model_best = training.train_model(
    model_best, loaders_final, cfg_final, epoch_hook=None, A_bin=A_bin,
    show_epoch_bar=True, leave_bar=True, progress_desc="Final (best GNN)"
)

# 6) Metrics (masked MAE/RMSE)
def masked_metrics_np(yhat, y, m):
    err = (yhat - y)
    mae = (np.abs(err) * m).sum() / (m.sum() + 1e-8)
    rmse = math.sqrt(((err**2) * m).sum() / (m.sum() + 1e-8))
    return float(mae), float(rmse)

# Baseline hist-avg
hist_pred = training.hist_avg_predict_masked(X, M, pd.DatetimeIndex(times), masks_final, cfg_final.H_in, cfg_final.H_out, ds_test=ds_test)
mae0, rmse0 = masked_metrics_np(hist_pred, Y_fut_test, M_fut_test)

# Linear
mae_lin, rmse_lin = masked_metrics_np(lin_pred, Y_fut_test, M_fut_test)

# GNN
test_metrics = training.evaluate(model_best, loaders_final["test"], cfg_final.device)
mae_gnn, rmse_gnn = float(test_metrics["MAE"]), float(test_metrics["RMSE"])

# 7) Summary table (% of baseline MAE)
def pct(x, base): return 100.0 * (x / base)

comp = pd.DataFrame([
    {"model": "Historical average",      "MAE": mae0,   "RMSE": rmse0,   "% of baseline MAE": 100.0},
    {"model": "Linear (exog_best)",      "MAE": mae_lin,"RMSE": rmse_lin,"% of baseline MAE": pct(mae_lin, mae0)},
    {"model": "GNN (exog_best + cfg*)",  "MAE": mae_gnn,"RMSE": rmse_gnn,"% of baseline MAE": pct(mae_gnn, mae0)},
]).sort_values("MAE").reset_index(drop=True)

display(comp)


# Visualisation

#### Training vs validation MAE per epoch

In [ ]:
# Purpose: training curves
hist = model_best.history
plt.figure(figsize=(6,4))
plt.plot(hist["epoch"], hist["train_mae"], label="train MAE (plain)")
plt.plot(hist["epoch"], hist["val"],       label="val MAE (plain)")
# Optional: also see the optimized objective
# plt.plot(hist["epoch"], hist["train_obj"], label="train objective", linestyle="--")
plt.xlabel("epoch"); plt.ylabel("MAE"); plt.legend(); plt.grid(True); plt.show()


#### HA vs linear regression vs best GNN

In [ ]:
# === Force specific days & plot Observed vs {HistAvg, Linear, GNN} on edge B->D ===

# Resolve the directed edge index for B->D from network_cfg
def resolve_edge_id(ncfg, u="B", v="D"):
    U, V = str(u), str(v)

    # 1) Prefer an explicit mapping if present
    if hasattr(ncfg, "edge_index") and isinstance(ncfg.edge_index, dict) and ncfg.edge_index:
        for (a, b), i in ncfg.edge_index.items():
            if str(a) == U and str(b) == V:
                return int(i)

    # 2) Fallback: search the directed edge list
    for attr in ("directed_edges", "edges_dir", "dir_edges", "edges"):
        if hasattr(ncfg, attr) and getattr(ncfg, attr) is not None:
            arr = getattr(ncfg, attr)
            for i, (a, b) in enumerate(arr):
                if str(a) == U and str(b) == V:
                    return int(i)

    raise ValueError(f"Could not find directed edge {U}->{V} in network_cfg.")

edge_id = resolve_edge_id(network_cfg, "B", "D")
print("edge_id (B->D) =", edge_id)

# -----------------------------
# 1) Exclusive day categories
# -----------------------------
flags_ = flags.reindex(pd.DatetimeIndex(times)).fillna(0).astype(bool)
is_wend = flags_["is_weekend"].to_numpy()
is_holi = flags_["is_holiday"].to_numpy()
is_exam = flags_["is_exam"].to_numpy()

weekday_only = (~is_wend) & (~is_holi) & (~is_exam)
weekend_only = ( is_wend) & (~is_holi) & (~is_exam)
exam_only    = ( is_exam) & (~is_wend) & (~is_holi)

# -----------------------------
# 2) Given a calendar date, return indices for that FULL day,
#    constrained to TEST split AND the chosen category
# -----------------------------
def indices_for_date(date_like, split_mask_bool, day_mask_bool):
    date = pd.Timestamp(date_like).normalize()
    idx_all = np.where(split_mask_bool & day_mask_bool)[0]  # indices in TEST & category
    days_all = pd.DatetimeIndex(times[idx_all]).normalize()
    sel = idx_all[days_all == date]
    if sel.size == 0:
        avail = pd.to_datetime(np.unique(days_all)).strftime("%Y-%m-%d").tolist()
        raise ValueError(
            f"No data for date {date.date()} in this category + TEST. "
            f"Available dates here: {avail[:15]}{' …' if len(avail)>15 else ''}"
        )
    return sel, date.date()

# -----------------------------
# 3) Collapse windowed predictions to a per-timestamp day series
#    (averaging contributions when multiple windows hit the same time)
# -----------------------------
def _collapse_to_day_series(day_idx, edge_id, ds_test, model_best, cfg_final,
                            hist_pred, lin_pred):
    """
    Returns (t_concat, obs, hist, lin, gnn) each length len(day_idx).
    - obs  : raw X on that day for edge_id
    - hist : averaged hist baseline at each timestamp
    - lin  : averaged linear baseline at each timestamp
    - gnn  : averaged GNN prediction at each timestamp (computed on-the-fly)
    """
    if day_idx is None or len(day_idx) == 0:
        return None

    pos = {int(g): i for i, g in enumerate(day_idx)}
    L = len(day_idx)
    E = X.shape[1]

    # observed series
    obs = X[day_idx, edge_id].astype(np.float32)

    # accumulators
    hist_sum = np.zeros(L, np.float32); hist_cnt = np.zeros(L, np.float32)
    lin_sum  = np.zeros(L, np.float32); lin_cnt  = np.zeros(L, np.float32)
    gnn_sum  = np.zeros(L, np.float32); gnn_cnt  = np.zeros(L, np.float32)

    # which batches start inside this day
    batch_ids = []
    for i in range(len(ds_test)):
        t0 = int(ds_test.starts[i]); t1 = t0 + cfg_final.H_in
        if t1 in pos:
            batch_ids.append(i)

    # roll predictions to the day grid
    model_best.eval()
    with torch.no_grad():
        for bi in batch_ids:
            item = ds_test[bi]
            t0 = int(ds_test.starts[bi]); t1 = t0 + cfg_final.H_in

            # hist + linear from precomputed tensors
            H = hist_pred.shape[1]
            for h in range(H):
                gidx = t1 + h
                if gidx in pos:
                    j = pos[gidx]
                    hist_sum[j] += float(hist_pred[bi, h, edge_id]); hist_cnt[j] += 1.0
                    lin_sum[j]  += float(lin_pred[bi, h,  edge_id]); lin_cnt[j]  += 1.0

            # GNN fresh forward pass for exact alignment
            xb = torch.from_numpy(item["x"].numpy()[None]).to(cfg_final.device)
            yh = model_best(xb).cpu().numpy()[0]  # [H_out,E]
            H = yh.shape[0]
            for h in range(H):
                gidx = t1 + h
                if gidx in pos:
                    j = pos[gidx]
                    gnn_sum[j] += float(yh[h, edge_id]); gnn_cnt[j] += 1.0

    def _safe_avg(s, c):
        out = np.zeros_like(s)
        mask = c > 0
        out[mask] = s[mask] / c[mask]
        return out

    hist = _safe_avg(hist_sum, hist_cnt)
    lin  = _safe_avg(lin_sum,  lin_cnt)
    gnn  = _safe_avg(gnn_sum,  gnn_cnt)
    t_concat = pd.DatetimeIndex(times[day_idx])

    return t_concat, obs, hist, lin, gnn

# -----------------------------
# 4) Force your chosen dates (YYYY-MM-DD)
# -----------------------------
idx_weekday, date_weekday = indices_for_date("2025-02-11", masks_final["test"], weekday_only)
idx_weekend, date_weekend = indices_for_date("2025-01-19", masks_final["test"], weekend_only)
idx_exam,    date_exam    = indices_for_date("2025-01-29", masks_final["test"], exam_only)

# Precompute all day series once (saves time and lets us share y-limits)
ds_test = loaders_final["test"].dataset
out_weekday = _collapse_to_day_series(idx_weekday, edge_id, ds_test, model_best, cfg_final, hist_pred, lin_pred)
out_weekend = _collapse_to_day_series(idx_weekend, edge_id, ds_test, model_best, cfg_final, hist_pred, lin_pred)
out_exam    = _collapse_to_day_series(idx_exam,    edge_id, ds_test, model_best, cfg_final, hist_pred, lin_pred)

def _maxy(out_tuple):
    if out_tuple is None: return 0.0
    _, obs, hist, lin, gnn = out_tuple
    return float(np.nanmax([obs.max() if len(obs) else 0,
                            hist.max() if len(hist) else 0,
                            lin.max()  if len(lin)  else 0,
                            gnn.max()  if len(gnn)  else 0]))

ymax = 1.05 * max(_maxy(out_weekday), _maxy(out_weekend), _maxy(out_exam))
ylim = (0, ymax if np.isfinite(ymax) and ymax > 0 else 1.0)

# -----------------------------
# 5) Plotting with shared y-axis
# -----------------------------
def plot_day_from_tuple(out_tuple, title_prefix, date_obj, ylim=None):
    if out_tuple is None:
        print(f"No data available for {title_prefix} ({date_obj}).")
        return
    t, obs, hist, lin, gnn = out_tuple
    fig, ax = plt.subplots(figsize=(9, 3.0))
    ax.plot(t, obs, label="Observed", linewidth=2)
    ax.plot(t, hist, label="Historical avg", alpha=0.9)
    ax.plot(t, lin,  label="Linear (exog)",   alpha=0.9)
    ax.plot(t, gnn,  label="GNN (best)",      alpha=0.9)
    ax.set_title(f"{title_prefix} — edge B→D — {date_obj}")
    ax.set_xlabel("Time"); ax.set_ylabel("Count"); ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", ncols=2, fontsize=9)
    if ylim is not None: ax.set_ylim(*ylim)
    ax.xaxis.set_major_formatter(DateFormatter("%H:%M"))
    plt.tight_layout(); plt.show()

plot_day_from_tuple(out_weekday, "Weekday", date_weekday, ylim=ylim)
plot_day_from_tuple(out_weekend, "Weekend", date_weekend, ylim=ylim)
plot_day_from_tuple(out_exam,    "Exam",    date_exam,    ylim=ylim)


#### Network at certain time

In [ ]:
# === Point-in-time network visualization: TRUE / PRED / ERROR  ===
# Uses your existing objects: network_cfg, times, X, M, loaders_final, model_best, cfg_final

# ----------------- CONFIG -----------------
day_str       = "2025-02-11"   # YYYY-MM-DD
time_str      = "08:45"        # local time within that day
use_closest_h = True           # if multiple windows hit ts: True->smallest h, False->largest
# Visual scaling (keep as-is or tweak)
power_gamma   = 0.50           # <1 spreads low values
clip_hi_pct   = 99.5           # tame outliers for vmax when auto-scaling

# ----------------- helpers: timestamp + data extraction -----------------
def _closest_global_timestamp(times_idx: pd.DatetimeIndex, day_str: str, time_str: str) -> pd.Timestamp:
    times_idx = pd.DatetimeIndex(times_idx)
    t_req = pd.Timestamp(f"{day_str} {time_str}")
    day_mask = (times_idx.date == pd.Timestamp(day_str).date())
    if not day_mask.any():
        raise ValueError(f"No timeline entries for day {day_str}.")
    day_times = times_idx[day_mask]
    k = int(np.argmin(np.abs(day_times.values.astype("datetime64[ns]") - np.datetime64(t_req))))
    t_sel = pd.Timestamp(day_times.values[k])
    if t_sel != t_req:
        print(f"Using closest timestamp on {day_str}: requested {t_req.time()}, selected {t_sel.time()}")
    return t_sel

def _per_edge_true_at(times_idx, X, M, t_sel):
    """Return dict[(u,v)]=true value (NaN if unobserved) at t_sel, and the global index."""
    times_idx = pd.DatetimeIndex(times_idx)
    if t_sel not in times_idx:
        raise ValueError("Selected timestamp not in global timeline.")
    gi = int(np.where(times_idx == t_sel)[0][0])  # global index
    vals = X[gi, :].astype(np.float32)           # [E]
    obs  = (M[gi, :] > 0.5)
    out = {}
    for i, (u, v) in enumerate(network_cfg.directed_edges):
        out[(u, v)] = float(vals[i]) if bool(obs[i]) else np.nan
    return out, gi

def _per_edge_pred_at(model, ds_test, device, gi_target, H_in, use_closest=True):
    """
    For global index gi_target, scan test windows and pick the prediction that hits it.
    If multiple windows hit, keep min(h) if use_closest else max(h). Returns dict[(u,v)]=pred.
    """
    model.eval()
    E = len(network_cfg.directed_edges)
    best_h = np.full(E, (9999 if use_closest else -9999), dtype=int)
    best_y = np.full(E, np.nan, dtype=np.float32)
    with torch.no_grad():
        for i in range(len(ds_test)):
            s = int(ds_test.starts[i]); t1 = s + H_in
            h = gi_target - t1
            if h < 0 or h >= ds_test.H_out: 
                continue
            xb = ds_test[i]["x"].numpy()[None]                  # [1,H_in,E,F]
            yb = model(torch.from_numpy(xb).to(device)).cpu().numpy()[0]  # [H_out,E]
            yh = yb[h, :]                                       # [E]
            take = (h < best_h) if use_closest else (h > best_h)
            idx = np.where(take)[0]
            if idx.size:
                best_h[idx] = h
                best_y[idx] = yh[idx].astype(np.float32)
    return {ev: (float(best_y[e]) if np.isfinite(best_y[e]) else np.nan)
            for e, ev in enumerate(network_cfg.directed_edges)}

# ----------------- helpers: curved reciprocal arrows + plotting -----------------
def _edge_curvature_map(directed_edges, rad=0.12):
    """
    Curvature map for every directed edge.
    Reciprocal pairs get opposite curvature (+rad / -rad); singletons get 0.0.
    Deterministic based on node-name ordering.
    """
    curv = {}
    # group by undirected pair
    undirected = {}
    for u, v in directed_edges:
        key = tuple(sorted((str(u), str(v))))
        undirected.setdefault(key, []).append((u, v))
    for key, dir_list in undirected.items():
        if len(dir_list) == 2:
            (u1, v1), (u2, v2) = dir_list
            a, b = key  # a < b (strings)
            # (a->b) gets +rad, (b->a) gets -rad
            if str(u1) == a and str(v1) == b:
                curv[(u1, v1)] = +rad
                curv[(u2, v2)] = +rad
            else:
                curv[(u1, v1)] = +rad
                curv[(u2, v2)] = +rad
        else:
            (u, v) = dir_list[0]
            curv[(u, v)] = 0.0
    for e in directed_edges:
        curv.setdefault(e, 0.0)
    return curv

def _plot_network_values_better(POS, edges, values, title,
                                *, norm_mode="power", gamma=0.5,
                                clip_hi_pct=99.5, vmin=None, vmax=None,
                                cmapname='viridis', ax=None):
    """
    Draw directed network with improved low-end separation and *curved* reciprocal arrows.
    values: dict[(u,v)] -> scalar (NaN allowed)
    """
    if ax is None: ax = plt.gca()
    G = nx.DiGraph(); G.add_nodes_from(POS.keys()); G.add_edges_from(edges)
    edgelist = list(edges)

    vals = np.array([values.get((u, v), np.nan) for (u, v) in edgelist], dtype=float)
    finite = np.isfinite(vals)
    lo = float(np.nanmin(vals)) if vmin is None else vmin
    if vmax is None:
        hi_raw = float(np.nanmax(vals)) if finite.any() else 1.0
        hi_pct = float(np.nanpercentile(vals[finite], clip_hi_pct)) if finite.any() else hi_raw
        hi = min(hi_raw, hi_pct)
    else:
        hi = vmax
    if not np.isfinite(lo): lo = 0.0
    if (not np.isfinite(hi)) or hi <= lo: hi = lo + 1.0

    if norm_mode == "power":
        norm = mcolors.PowerNorm(gamma=gamma, vmin=lo, vmax=hi)
    elif norm_mode == "linear":
        norm = mcolors.Normalize(vmin=lo, vmax=hi)
    else:
        # optional: log scaling if ever needed
        norm = mcolors.LogNorm(vmin=max(lo, 1e-3), vmax=max(hi, 1e-3))
    cmap = cm.get_cmap(cmapname)

    colors = [cmap(norm(v)) if np.isfinite(v) else (0.85, 0.85, 0.85, 1.0) for v in vals]
    widths = np.full_like(vals, 2.0, dtype=float)
    curv   = _edge_curvature_map(edgelist, rad=0.12)
    styles = [f"arc3,rad={curv[(u, v)]}" for (u, v) in edgelist]

    ax.set_title(title)
    nx.draw_networkx_nodes(G, POS, node_size=300, node_color='lightgray',
                           edgecolors='k', linewidths=0.5, ax=ax)
    for (u, v), col, w, cs in zip(edgelist, colors, widths, styles):
        nx.draw_networkx_edges(
            G, POS, edgelist=[(u, v)], width=w, edge_color=[col],
            arrows=True, arrowstyle='-|>', arrowsize=10,
            connectionstyle=cs, ax=ax
        )
    nx.draw_networkx_labels(G, POS, font_size=9, ax=ax)

    sm = cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    cbar = ax.figure.colorbar(sm, ax=ax, shrink=0.78)
    cbar.set_label('value')
    ax.set_axis_off()
    return lo, hi, norm

# ----------------- MAIN -----------------
# 1) pick timestamp (on your global timeline)
t_sel = _closest_global_timestamp(pd.DatetimeIndex(times), day_str, time_str)

# 2) TRUE at that time (+ global index)
edge_true, gi = _per_edge_true_at(pd.DatetimeIndex(times), X, M, t_sel)

# 3) PRED from your trained model on the TEST dataset
ds_test  = loaders_final["test"].dataset
edge_pred = _per_edge_pred_at(model_best, ds_test, cfg_final.device, gi, cfg_final.H_in, use_closest=use_closest_h)

# 4) ERROR = pred - true (only where true is observed)
edge_err = {}
for (u, v) in network_cfg.directed_edges:
    tru = edge_true[(u, v)]; prd = edge_pred[(u, v)]
    edge_err[(u, v)] = (prd - tru) if (np.isfinite(tru) and np.isfinite(prd)) else np.nan

# 5) shared scales for TRUE/PRED; symmetric for ERROR
tp_vals = [v for v in (*edge_true.values(), *edge_pred.values()) if np.isfinite(v)]
tp_max  = float(np.nanpercentile(tp_vals, clip_hi_pct)) if tp_vals else 1.0
tp_max  = tp_max if tp_max > 0 else 1.0

err_vals = [abs(v) for v in edge_err.values() if np.isfinite(v)]
err_max  = float(np.nanpercentile(err_vals, clip_hi_pct)) if err_vals else 1.0
err_max  = max(err_max, 1.0)

# 6) draw panels (curved, separated arrows)
fig, axs = plt.subplots(3, 1, figsize=(5, 16), constrained_layout=True)

_plot_network_values_better(
    network_cfg.pos, network_cfg.directed_edges, edge_true,
    f"TRUE @ {pd.Timestamp(t_sel).strftime('%Y-%m-%d %H:%M')}",
    norm_mode="power", gamma=power_gamma, clip_hi_pct=clip_hi_pct,
    vmin=0, vmax=tp_max, cmapname='viridis', ax=axs[0]
)

_plot_network_values_better(
    network_cfg.pos, network_cfg.directed_edges, edge_pred,
    f"PRED @ {pd.Timestamp(t_sel).strftime('%Y-%m-%d %H:%M')} (h={'min' if use_closest_h else 'max'})",
    norm_mode="power", gamma=power_gamma, clip_hi_pct=clip_hi_pct,
    vmin=0, vmax=tp_max, cmapname='viridis', ax=axs[1]
)

_plot_network_values_better(
    network_cfg.pos, network_cfg.directed_edges, edge_err,
    f"ERROR (pred–true, observed only) @ {pd.Timestamp(t_sel).strftime('%Y-%m-%d %H:%M')}",
    norm_mode="linear", vmin=-err_max, vmax=err_max, cmapname='coolwarm', ax=axs[2]
)

plt.show()
